### Consumindo a API

In [1]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# URL com histórico real de 7 dias + previsão de 7 dias
url = "https://api.open-meteo.com/v1/forecast?latitude=-22.90&longitude=-43.20&hourly=temperature_2m,precipitation,windspeed_10m&timezone=auto&past_days=7"

try:
    response = requests.get(url)
    response.raise_for_status() 
    dados_json = response.json()
    
    dados = pd.DataFrame(dados_json['hourly']) #convertendo para df

except Exception:
    print("Erro na extração")

### Tratamento de dados

In [ ]:
# .copy() para não alterar os dados originais
df = dados.copy()

time              0
temperature_2m    0
precipitation     0
windspeed_10m     0
dtype: int64


In [ ]:
# entendendo os valores nulos
nulos = df.isna().sum() 
print(nulos)

##tratamento preventivo
df = df.dropna()

In [3]:
print(df)

                 time  temperature_2m  precipitation  windspeed_10m
0    2026-03-12T00:00            22.9            0.4            0.8
1    2026-03-12T01:00            22.8            0.4            1.1
2    2026-03-12T02:00            22.7            0.5            0.8
3    2026-03-12T03:00            22.5            0.2            0.5
4    2026-03-12T04:00            22.5            0.0            0.4
..                ...             ...            ...            ...
331  2026-03-25T19:00            26.4            0.0            6.7
332  2026-03-25T20:00            25.9            0.0            4.3
333  2026-03-25T21:00            25.4            0.0            2.1
334  2026-03-25T22:00            24.9            0.0            1.1
335  2026-03-25T23:00            24.5            0.0            1.6

[336 rows x 4 columns]


In [4]:
#convertendo a coluna de tempo
df['time'] = pd.to_datetime(df['time'])
display(df.head())

,time,temperature_2m,precipitation,windspeed_10m
0,2026-03-12 00:00:00,22.9,0.4,0.8
1,2026-03-12 01:00:00,22.8,0.4,1.1
2,2026-03-12 02:00:00,22.7,0.5,0.8
3,2026-03-12 03:00:00,22.5,0.2,0.5
4,2026-03-12 04:00:00,22.5,0.0,0.4


In [5]:
##traduzindo os meses (por padrão é em inglês)
traducao_meses = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}

# criando a coluna 'mes'
df['mes'] = df['time'].dt.month.map(traducao_meses)
display(df[['time', 'mes']].head())

,time,mes
0,2026-03-12 00:00:00,Março
1,2026-03-12 01:00:00,Março
2,2026-03-12 02:00:00,Março
3,2026-03-12 03:00:00,Março
4,2026-03-12 04:00:00,Março


In [ ]:
# organização e renomeação

df = df.rename(columns={
    'time': 'data',
    'temperature_2m': 'temperatura',
    'precipitation': 'precipitacao',
    'windspeed_10m': 'velocidade_vento'
})

## Manipulação de datas e feature engineer


In [7]:
## filtrar últimos 7 dias
hoje = datetime.now()
sete_dias_atras = hoje - timedelta(days=7)

df_filtrado = df[(df['data'] >= sete_dias_atras) & (df['data'] <= hoje)].copy()

# média de temperatura
df['media_temp_diaria'] = df.groupby(df['data'].dt.date)['temperatura'].transform('mean').round(1)


print(f"Lógica de filtro: De {sete_dias_atras.date()} até {hoje.date()}")
display(df[['data', 'temperatura', 'media_temp_diaria', 'precipitacao']].head())

Lógica de filtro: De 2026-03-12 até 2026-03-19


,data,temperatura,media_temp_diaria,precipitacao
0,2026-03-12 00:00:00,22.9,24.3,0.4
1,2026-03-12 01:00:00,22.8,24.3,0.4
2,2026-03-12 02:00:00,22.7,24.3,0.5
3,2026-03-12 03:00:00,22.5,24.3,0.2
4,2026-03-12 04:00:00,22.5,24.3,0.0


In [8]:
# criando definições do tempo
regras = [
    (df['precipitacao'] > 0.5),                                # se chove mais de 0.5mm
    (df['precipitacao'] > 0) & (df['precipitacao'] <= 0.5),    # se chove só um pouquinho
    (df['temperatura'] > 26) & (df['precipitacao'] == 0),      # calor e sem chuva
    (df['temperatura'] <= 26) & (df['precipitacao'] == 0)      # fresco e sem chuva
]

# rótulos que quero no BI pra coluna de condição
resultados = ['Tempestade', 'Chuvoso', 'Ensolarado', 'Nublado'] 

# coluna de condição
df['condicao'] = np.select(regras, resultados, default='Céu Limpo')

display(df[['data', 'temperatura', 'precipitacao', 'condicao']].head(10))

,data,temperatura,precipitacao,condicao
0,2026-03-12 00:00:00,22.9,0.4,Chuvoso
1,2026-03-12 01:00:00,22.8,0.4,Chuvoso
2,2026-03-12 02:00:00,22.7,0.5,Chuvoso
3,2026-03-12 03:00:00,22.5,0.2,Chuvoso
4,2026-03-12 04:00:00,22.5,0.0,Nublado
5,2026-03-12 05:00:00,22.5,0.0,Nublado
6,2026-03-12 06:00:00,22.5,0.0,Nublado
7,2026-03-12 07:00:00,22.8,0.0,Nublado
8,2026-03-12 08:00:00,23.3,0.0,Nublado
9,2026-03-12 09:00:00,23.8,0.1,Chuvoso


In [9]:
##categorizando a intensidade do vento

def desc_vento(v):
    if v < 1: return 'Calmo'
    elif v < 10: return 'Brisa Leve'
    elif v < 20: return 'Vento Moderado'
    else: return 'Vento Forte'

df['classificacao_vento'] = df['velocidade_vento'].apply(desc_vento)

In [10]:
# definindo é horário de pico/comercial (ex: 08h as 18h)
df['hora'] = df['data'].dt.hour
# cria a coluna 'horario_comercial'
df['horario_comercial'] = df['hora'].apply(lambda x: 'Comercial' if 8 <= x <= 18 else 'Fora de Hora')

In [11]:
## criando o periodo do dia
def definir_periodo(h):
    if 6 <= h < 12: return 'Manhã'
    elif 12 <= h < 18: return 'Tarde'
    elif 18 <= h < 24: return 'Noite'
    else: return 'Madrugada'

df['periodo'] = df['hora'].apply(definir_periodo)

print(df[['hora', 'periodo']].head())

   hora    periodo
0     0  Madrugada
1     1  Madrugada
2     2  Madrugada
3     3  Madrugada
4     4  Madrugada


In [12]:
#criando estações do ano

def definir_estacao(data):
    dia = data.day
    mes = data.month
    if (mes == 12 and dia >= 21) or (mes in [1, 2]) or (mes == 3 and dia < 20):
        return 'Verão'
    elif (mes == 3 and dia >= 20) or (mes in [4, 5]) or (mes == 6 and dia < 21):
        return 'Outono'
    elif (mes == 6 and dia >= 21) or (mes in [7, 8]) or (mes == 9 and dia < 22):
        return 'Inverno'
    else:
        return 'Primavera'
df['estacao'] = df['data'].apply(definir_estacao)

In [13]:
print("Colunas atuais no DataFrame:", df.columns.tolist())

Colunas atuais no DataFrame: ['data', 'temperatura', 'precipitacao', 'velocidade_vento', 'mes', 'media_temp_diaria', 'condicao', 'classificacao_vento', 'hora', 'horario_comercial', 'periodo', 'estacao']


## Organização dos dados

#### Alternativa 1 - (Usada)

In [14]:
nome_pasta = "analise_previsao"

#diretorio
os.makedirs(nome_pasta, exist_ok=True)

## caminho do arquivo
caminho_arquivo = os.path.join(nome_pasta, "dados.csv")

# salvando em CSV
df.to_csv(caminho_arquivo, index=False, encoding='utf-8-sig', sep=';', decimal=',') ##precisei acrescentar uma ',' antes, pois o PBI estava modificando o tipo de dado

print(f"o arquivo foi salvo em: {caminho_arquivo}")

o arquivo foi salvo em: analise_previsao\dados.csv


### Alternativa 2 - (Usável)


In [15]:
import sqlite3

# conectando o banco no arquivo que criei
conn = sqlite3.connect("analise_previsao/db.db")

# 2. ENVIA os dados do seu DataFrame (df_final) para o banco
df.to_sql('previsao_tempo', conn, if_exists='replace', index=False)

# 3. Agora sim, você faz a PERGUNTA (Sua consulta)
query = "SELECT * FROM previsao_tempo"
resumo_sql = pd.read_sql(query, conn)

print("Resumo gerado via SQL:")
display(resumo_sql.head()) # Use display para ficar mais bonito no VS Code

# 4. Fecha a conexão
conn.close()

Resumo gerado via SQL:


,data,temperatura,precipitacao,velocidade_vento,mes,media_temp_diaria,condicao,classificacao_vento,hora,horario_comercial,periodo,estacao
0,2026-03-12 00:00:00,22.9,0.4,0.8,Março,24.3,Chuvoso,Calmo,0,Fora de Hora,Madrugada,Verão
1,2026-03-12 01:00:00,22.8,0.4,1.1,Março,24.3,Chuvoso,Brisa Leve,1,Fora de Hora,Madrugada,Verão
2,2026-03-12 02:00:00,22.7,0.5,0.8,Março,24.3,Chuvoso,Calmo,2,Fora de Hora,Madrugada,Verão
3,2026-03-12 03:00:00,22.5,0.2,0.5,Março,24.3,Chuvoso,Calmo,3,Fora de Hora,Madrugada,Verão
4,2026-03-12 04:00:00,22.5,0.0,0.4,Março,24.3,Nublado,Calmo,4,Fora de Hora,Madrugada,Verão


In [16]:
df.columns

Index(['data', 'temperatura', 'precipitacao', 'velocidade_vento', 'mes',
       'media_temp_diaria', 'condicao', 'classificacao_vento', 'hora',
       'horario_comercial', 'periodo', 'estacao'],
      dtype='object')

In [17]:
##teste

with sqlite3.connect("analise_previsao/db.db") as conn:
    query_chuva = "SELECT data, periodo, precipitacao FROM previsao_tempo WHERE precipitacao > 0"
    df_alertas = pd.read_sql(query_chuva, conn)
    
    print(df_alertas if not df_alertas.empty else "Nenhuma chuva encontrada.")

                   data    periodo  precipitacao
0   2026-03-12 00:00:00  Madrugada           0.4
1   2026-03-12 01:00:00  Madrugada           0.4
2   2026-03-12 02:00:00  Madrugada           0.5
3   2026-03-12 03:00:00  Madrugada           0.2
4   2026-03-12 09:00:00      Manhã           0.1
5   2026-03-12 10:00:00      Manhã           0.2
6   2026-03-12 13:00:00      Tarde           0.5
7   2026-03-12 14:00:00      Tarde           0.7
8   2026-03-12 15:00:00      Tarde           0.2
9   2026-03-12 16:00:00      Tarde           0.1
10  2026-03-12 19:00:00      Noite           0.3
11  2026-03-12 20:00:00      Noite           0.1
12  2026-03-12 21:00:00      Noite           0.9
13  2026-03-12 22:00:00      Noite           1.7
14  2026-03-12 23:00:00      Noite           0.8
15  2026-03-13 05:00:00  Madrugada           0.2
16  2026-03-13 06:00:00      Manhã           0.6
17  2026-03-13 07:00:00      Manhã           0.3
18  2026-03-13 09:00:00      Manhã           0.7
19  2026-03-13 12:00